# 01 — The three question paradigms
- **Noul**: probability that a statement is true.
- **Choice**: probability of each declared option (exactly one correct).
- **Score**: probability of each level of an ordered rubric.

Questions can be batched into a single `/v1/systemone` request.


In [ ]:
from decision_qa import to_systemone_payload, NoulQuestion, ChoiceQuestion, ScoreQuestion

qs = [
    NoulQuestion(label='is_true', statement='Paris is the capital of France.'),
    ChoiceQuestion(label='color', options=['red', 'green', 'blue']),
    ScoreQuestion(label='quality', rubric='Rate the answer quality.',
                  levels=['poor', 'fair', 'good']),
]
import json
print(json.dumps(to_systemone_payload({'question': 'Example'}, qs), indent=2))


## Confidence and abstention
A decision is only useful if we know when to trust it. A simple policy: commit when the top probability clears a threshold, otherwise escalate to a stronger model or a human.


In [ ]:
def act(result, threshold=0.8):
    label = max(result.probabilities, key=result.probabilities.get)
    p = result.probabilities[label]
    return (label, 'commit') if p >= threshold else (label, 'escalate')

from decision_qa import MockClient
client = MockClient(seed=1)
res = client.decision({'question': 'Example'}, qs)[0]
print(res.probabilities, '->', act(res))


## Order invariance
A good decision model should give the same answer when option order changes. Use `option_order_invariance` with a labelled question; an unlabelled mock returns uniform probabilities, so ties make the 'top' answer arbitrary.


In [ ]:
from decision_qa.evaluate import option_order_invariance
from decision_qa import ChoiceQuestion
labelled = ChoiceQuestion(label='color', options=['red', 'green', 'blue'], gold_index=1)
print('order invariant:', option_order_invariance(client, {'question': 'Example'}, labelled))


Free further reading: [awesome-jev-prompts](https://github.com/vicfei/awesome-jev-prompts) (51 field-tested question patterns, CC0) and the [concept docs](https://docs.typesafe.ai/introduction).
